# Notebook 4 <br>Reshaping data and time series

In the previous notebooks you saw how to inspect data, repair it, check it
systematically and combine different tables. We always assumed that it had the right *shape* (correct rows and columns).

Often that isn't the case, so what do we do then?

Start by loading pandas:

In [ ]:
import pandas as pd
import numpy as np
import pooch

%reload_ext autoreload
%autoreload 2

pd.set_option("display.precision", 2)

This notebook needs a few extra files: the data you will work with, and `tests.py`, which checks your answers. Run the cell below to download them into the folder this notebook is in. You only need to do this once; after that the files are already on your computer and the cell finishes immediately.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/spcourse/pandas/main/pandas-dp/data/"

print("downloading data files")
for fname in ["growth_2026.csv", "tests.py"]:
    pooch.retrieve(url=DATA_URL + fname, known_hash=None, fname=fname, path=".", progressbar=True)
print("done!")

from tests import *

## Chapter 12: Reshaping data

For most kinds of data analyses, plots, and machine learning purposes we want data to follow a specific pattern: every column is one variable (or feature), every row represents an observation and each cell contains a single value.

But a lot of datasets do not provide data that way.

Let's look at an example. This is synthetic data (made up), but it has the same shape you would get from the World Bank. It gives the inflation per country over 2021, 2022 and 2023. It also encodes the economic blocs that the countries are part of (e.g., Spain is part of the European Union (EU) and the Organisation for Economic Co-operation and Development (OECD)).

In [ ]:
econ = pd.DataFrame({
    "country": ["Netherlands", "Spain", "Germany", "Norway", "Brazil"],
    "blocs":   ["EU, OECD", "EU, OECD", "EU, OECD, G7", "OECD, EFTA", "G20"],
    "2021": [2.7, 3.0, 3.1, 3.5, 8.3],
    "2022": [11.6, 8.3, 8.7, 5.8, 9.3],
    "2023": [4.1, 3.4, 5.9, 5.5, 4.6],
})

econ

This data has a few problems for further processing. The years (`2021`, `2022` and `2023`) are columns, but should not be treated as variables, but as values (the variable is `year`).  And the cells for the variable `blocs` contain multiple values in a single cell (in the form of a list).

### Turning columns into values (melting)

The method `melt` turns the names of a set of columns into one column, and their values into
another. It's easier to understand by looking at it:

In [ ]:
long = econ.melt(
    id_vars=["country", "blocs"],
    var_name="year",
    value_name="inflation",
)

long["year"] = long["year"].astype(int)

display(long)

You see that the years (`2021`, `2022` and `2023`) have gone from column names to being values for a new column named `year`. You can see that a second new column has been introduced, called `inflation`. This corresponds to the original inflation numbers that were in the `2021`, `2022` and `2023` columns. 

Check the resulting table well, to convince yourself that no information has been lost by this transformation. It's just a different representation of the same data: five countries times three melted columns is fifteen rows.

The melt operation has three important parameters:

* `id_vars=["country", "blocs"]`, these are the names of the columns that should *not* be melted. Often you have a lot of columns to be melted (imagine this table for the last 100 years) so it is convenient that you can specify what not to melt, rather than what to melt.
* `var_name="year"`, this is the name of the new column that will contain the old column names as values.
* `value_name="inflation"`, this is the name of the new column that should contain the data from the melted columns.


Pay attention to the conversion of `year` to int. As the values in the `year` column were originally column names, they are still represented as strings. So you need to explicitly convert them if you need them to be another type (`int`, `float`, `datetime`, ...).

### Exercise 27

Below is another DataFrame with a similar shape as before. It contains the GDP growth per year (again, synthetic data). 

Melt it into a long frame called `gdp_long`, with the years in a column called `year` and
the numbers in a column called `growth`. Make sure to keep all information that is in the original DataFrame. The result should contain 4 columns and 9 rows.

In [ ]:
gdp = pd.DataFrame({
    "country": ["Netherlands", "Spain", "Germany"],
    "agency":  ["CBS", "INE", "Destatis"],
    "2021": [6.2, 6.4, 3.7],
    "2022": [4.3, 5.8, 1.4],
    "2023": [0.1, 2.5, -0.3],
})

display(gdp)

gdp_long = None

# YOUR CODE HERE

print("rows:", len(gdp_long))
display(gdp_long)

In [ ]:
test_27(gdp_long, gdp)

### The reverse (pivoting data)

The method `pivot` does the opposite. You have a table like `long` above and you turn it back into the wide shape:

In [ ]:
wide = long.pivot(
    index=["country", "blocs"], 
    columns="year", 
    values="inflation"
)

display(wide)

As you can see, this gives you back the original table.

The `pivot` operation has three important parameters:

* `index=["country", "blocs"]`, the row ids that should be preserved.  
* `columns="year"`, the values in this column (or these columns) will become the column names in the pivoted table.
* `values="inflation"`, these are the values that should populate the newly created columns.

For big data sets that you cannot inspect by eye it is good to do these sanity checks: If you did a pivot, melt it again and see if you got the same. And if you did a melt on your data, pivot it again and check. This can be a bit fiddly, though, because column names are subtly changed and the row order is different (pivot sorts the rows). So you would need to deal with that: 

In [ ]:
# fix 1: column names identical pivoted data
check_wide = wide.reset_index()
check_wide.columns = econ.columns # careful: this only works because the columns are in the same order   
display(check_wide)

# fix 2 sorting original data in the same way as pivoted data
check_econ = econ.sort_values("country").reset_index(drop=True)
display(check_econ)

print(f"metling + pivoting = original: {check_wide.equals(check_econ)}")

### Dealing with nested data (exploding)

There was another problem: the bloc information is nested. Meaning that `blocs` contains multiple values in each cell.

In [ ]:
display(econ[["blocs"]])

These are actually not even lists yet, but just strings that happen to look like lists. Just have a look at the dtype for `blocs`:

In [ ]:
display(econ.dtypes)

We need to unpack this: First we need to convert the string into an actual Python list. After we've done that, we can explode the data.

In [ ]:
tidy = econ.copy()

# 1. convert string into list
tidy["blocs"] = tidy["blocs"].str.split(", ")
display(tidy)

# 2. explode
tidy = tidy.explode("blocs")
display(tidy)

As you can see we have a separate row for each economic block a country belong to. Note that this creates a lot of duplicate information. The inflation numbers for the Netherlands, Spain and Norway occur twice, and for Germany even three times in this table. Which as we've seen before can be a problem in itself (there was a good reason that this information was nested in the original data). But often you need the data in this form for further analyses or visualizations.

Note: pay attention to a subtle but important detail: split on exactly the right separator. These values are separated by a comma and a
space, not just a comma. If you split on `","` instead of `", "` you get messy results:

In [ ]:
sloppy = econ.copy()
sloppy["blocs"] = sloppy["blocs"].str.split(",")
sloppy = sloppy.explode("blocs")

sloppy["blocs"].value_counts()

So now, for example, `"OECD"` and `" OECD"` are considered different values.

The `explode` method repeats index labels. So, for example, Germany's three blocs all carry Germany's original index `2`. This will mess with `.loc` (`tidy.loc[2]` now returns three rows instead of one) and can create complications down the road. So it's a good idea to reset the index after exploding:

In [ ]:
tidy = tidy.reset_index(drop=True)

display(tidy)

### Exercise 28

The `peaks` DataFrame below lists six mountains in the Alps: how high they are, and which
countries they lie in. Several of them sit on a border, so a cell in `countries` can hold
more than one value. Find the height of the highest peak of each country, as a Series called `highest`, indexed
by country. A peak on a border counts for both of its countries: Mont Blanc is the highest point of
France as well as of Italy, so it should turn up in both.

Code this in such a way that it would work just as well if we changed the data for another mountain range. (So no hardcoding information.)

In [ ]:
peaks = pd.DataFrame({
    "peak":      ["Mont Blanc", "Matterhorn", "Zugspitze", "Grossglockner",
                  "Marmolada", "Dufourspitze"],
    "countries": ["France; Italy", "Switzerland; Italy", "Germany; Austria",
                  "Austria", "Italy", "Switzerland"],
    "height_m":  [4806, 4478, 2962, 3798, 3343, 4634],
})

display(peaks)

highest = None

# YOUR CODE HERE

display(highest)

In [ ]:
test_28(highest, peaks)

### Pivoting with redundant data

When we have duplicate rows `pivot` will not work anymore. 

Suppose we want the inflation table back in its wide form, one row per country.
Everything we need is there: `melt` gave us a `year` column, and `explode` gave us one row
per bloc.

In [ ]:
by_bloc = long.copy()
by_bloc["blocs"] = by_bloc["blocs"].str.split(", ")
by_bloc = by_bloc.explode("blocs").reset_index(drop=True)

by_bloc

So, this is the fully expanded table. But if we would want to turn it into a wide form with `pivot` it would give an error. This is because for most countries we have duplicate values that indicate the inflation for a specific year (e.g., the fact that Germany had an inflation of 3.1% in 2021 is in there three times). `pivot` doesn't know which value to pick (they happen to be the same, so it doesn't matter, but Pandas doesn't know that).

So this will give an error:

In [ ]:
by_bloc.pivot(index="country", columns="year", values="inflation")

`ValueError: Index contains duplicate entries, cannot reshape`.

When this happens you can use `pivot_table` instead:

In [ ]:
by_bloc.pivot_table(index="country", columns="year", values="inflation")

The `pivot_table` method does the same thing as `pivot` but aggregates all values that land in the same
cell. The default aggregation function is `mean`, which is fine in this case, because all values are the same anyway. But of course this aggregation function can be configured.

This can be useful if we want to get aggregate information per economic bloc, instead of per country:

In [ ]:
print("Mean inflation per economic bloc (unweighted, all economies contribute equally to this mean):")
display(by_bloc.pivot_table(index="blocs", columns="year",
                            values="inflation", aggfunc="mean"))

print("Max inflation per economic bloc:")
display(by_bloc.pivot_table(index="blocs", columns="year",
                            values="inflation", aggfunc="max"))


### Exercise 29

The `rents` DataFrame below contains the average rent in two cities in each of two countries, over two
years (again, synthetic data). 

Build `rent_table`: one row per country, one column per year, holding the **mean** rent
across that country's cities.

The result should have this shape (the dots stand for the numbers you compute):

```
year       2022    2023
country
ES          ...     ...
NL          ...     ...
```

Note where the names `year` and `country` end up: they label the columns and the index, not
a cell. That is what `index=` and `columns=` do.

In [ ]:
rents = pd.DataFrame({
    "country": ["NL", "NL", "ES", "ES", "NL", "NL", "ES", "ES"],
    "city":    ["Amsterdam", "Rotterdam", "Madrid", "Seville"] * 2,
    "year":    [2022] * 4 + [2023] * 4,
    "rent":    [1780, 1310, 1120, 780, 1900, 1420, 1240, 850],
})

display(rents)
rent_table = None

# YOUR CODE HERE

display(rent_table)

In [ ]:
test_29(rent_table, rents)

### Summary

All the ways we've seen to transform DataFrames:

| input | output | use |
|---|---|---|
| a column per year | a `year` column | `melt(id_vars=...)` |
| several values in one cell | one per row | `str.split()` then `explode()` |
| long records | a wide table | `pivot(index=, columns=, values=)` |
| ...with more than one value per cell | a wide table | `pivot_table(..., aggfunc=)` |

Sanity checks to use:

- pivot + melt -> original table <- melt + pivot
- after `melt`, rows out should equal rows in times the number of melted columns
- after `explode`, run `value_counts()` on the exploded column

## Chapter 13: Groups and time

Here we will look at some issues you might run into when dealing with time series.

In time series, the rows have an order that matters and that affects the relevant operations. Think about inter row differences, running
totals, moving averages, ...

### The experiment

We use synthetic data again (making up data is just so convenient for highlighting specific issues you might run into :)).

We imagine an experiment that tries to measure the effect of fertiliser on plants. 3 plants were grown with fertiliser and 3 plants without, as our control group. We measured the height of each plant with a ruler at regular intervals.

In [ ]:
growth = pd.read_csv("growth_2026.csv", parse_dates=["day"])

print(growth.shape)
display(growth)

First let's do some basic sanity checks on this data:

In [ ]:
print("plants:    ", growth["plant_id"].nunique())
print("treatments:", list(growth["treatment"].unique()))
print("period:    ", growth["day"].min().date(), "->", growth["day"].max().date())
print("time span: ", (growth["day"].max().date() - growth["day"].min().date()).days)
print("heights:   ", growth["height_cm"].min(), "-", growth["height_cm"].max(), "cm")
print("missing:   ", growth.isna().sum().sum())
print()
print("measurements per plant:", growth.groupby("plant_id").size().unique())

It looks good: two treatment groups, 6 plants, reasonable heights and date ranges, etc.

There is one thing to note: the time span of the experiment is 88 days, but there are only 60 measurements per plant. This means that we don't have a measurement for each day. This will be important later.

### Comparing to the previous row

Let's look at a smaller subset of the data so it's easier to inspect the results:

In [ ]:
sample = pd.DataFrame({
    "day": pd.to_datetime(["2026-04-09", "2026-04-10", "2026-04-13", "2026-04-14"] * 2),
    "plant_id":  ["A1"] * 4 + ["B1"] * 4,
    "treatment": ["control"] * 4 + ["fertiliser"] * 4,
    "height_cm": [26.1, 26.8, 28.3, 29.2, 37.6, 39.9, 43.3, 44.2],
})

display(sample)

The function `shift` moves a column down by one row, so that each row can see the one before it. We can use that to compute, for example, how much a plant grew in one day.



In [ ]:
sample["previous"] = sample["height_cm"].shift()
sample["grew_by"]  = sample["height_cm"] - sample["previous"]
display(sample)

First shifting and then taking the difference is common enough that pandas has a built-in method that does the same in one call, `diff`:

In [ ]:
sample["grew_by"] = sample["height_cm"].diff()
display(sample)

Note that in either case `grew_by` starts with a `NaN`, which is good. In the first row we don't have a previous row to compare it to. 

Row 4 is more problematic. This should also be a `NaN`, because it is a different plant than the row above it. Plant B1 did not grow 8.4 cm on 2026-04-09. We actually don't know how much it grew as it is the first measurement of B1.

The problem is that `diff` blindly subtracts the previous *row*, and has no idea that the previous row belongs to a different plant.

We can fix this using `groupby`, because `diff()` can also be applied to groups:

In [ ]:
sample["grew_by"] = sample.groupby("plant_id")["height_cm"].diff()
display(sample)

Each plant's first row is `NaN` now, which is what we want.

### Row numbers are not the same as time.

The problem is that the `grew_by` number is still not that useful if we want to know the growth rate (the growth per day), as the rows do not directly correspond to single days. Look at the DataFrame above: 3 days passed between rows 5 and 6 (2026-04-10 and 2026-04-13), probably because the weekend was skipped. `diff` gives you the change per row, but what we would like is the change per day. 

We have the dates, so we can compute the gap in days per row, and use that to compute the growth per day.

In [ ]:
# compute the gap in days
sample["gap_days"] = sample.groupby("plant_id")["day"].diff().dt.days

# compute the actual growth rate
sample["growth_per_day"]  = sample["grew_by"] / sample["gap_days"]

display(sample)

This original dataset shows a lot of gaps:

### Exercise 30

The DataFrame `readings` below contains readings from a water-level gauge, read on an irregular schedule.

Add two columns to it: `rise`, the change in level since the previous reading, and
`rise_per_day`, that change divided by the number of days it took.

In [ ]:
readings = pd.DataFrame({
    "day":     pd.to_datetime(["2026-03-02", "2026-03-03", "2026-03-06",
                               "2026-03-07", "2026-03-14"]),
    "level_m": [1.20, 1.28, 1.55, 1.61, 2.10],
})

# YOUR CODE HERE

readings

In [ ]:
test_30(readings)

### Smoothing

The ruler used in this experiment is not a very precise tool. It measures to the nearest millimetre, which means that the potential measurement error is often larger than the growth rate. So the result of the `diff` operation will be very noisy. We can see that if we plot the daily growth rate of the original dataset:

In [ ]:
b1 = growth[growth["plant_id"] == "B1"].set_index("day")[["height_cm"]]
b1["growth_per_day"] = b1["height_cm"].diff() / b1.index.to_series().diff().dt.days

b1["growth_per_day"].plot(title="B1 daily growth, as measured", ylabel="cm/day")

You see that the growth rate is all over the place and, even worse, often negative. Did the plant shrink?

This is most likely the result of measurement error.

We can use a rolling window to replace each value with a summary of the values around it. So instead of using `growth_per_day` directly we use a rolling window to compute the average of the 7 days around it. You can use the `rolling` method for that:

In [ ]:
b1["smoothed"] = b1["growth_per_day"].rolling("7D").mean()

b1[["growth_per_day", "smoothed"]].plot(title="B1 daily growth", ylabel="cm/day");

You can directly see that the smoothed data looks much less erratic.

It also helps us to look a bit beyond the measurement errors. The maximum growth rate according to the original data is almost 3 cm/day; according to the smoothed data it is closer to 1.3 cm/day. The second is more likely to be close to the actual growth rate.

#### Some technical details

One detail in that call matters more than it looks: the window was `"7D"`, a string,
not `7`.

In [ ]:
b1["by_row"]  = b1["height_cm"].rolling(7).mean()
b1["by_time"] = b1["height_cm"].rolling("7D").mean()

print("NaN with rolling(7):   ", b1["by_row"].isna().sum())
print("NaN with rolling('7D'):", b1["by_time"].isna().sum())
print("largest disagreement:  ",
      (b1["by_row"] - b1["by_time"]).abs().max().round(2), "cm")

The first, `rolling(7)` uses a window of seven **rows**. The second, `rolling("7D")` uses a window of seven **days**. This only works if the frame is indexed by date, which is why `b1` was given a `DatetimeIndex` above. 

It can be useful to look at what `rolling` produces in a bit more detail. Let's create a rolling window of 4 rows:

In [ ]:
rolling = b1["growth_per_day"].rolling(4)
print(rolling)

It creates a `Rolling` object, that in many ways behaves like a groupby object. We can iterate over it to inspect them. So let's show the first 4 windows:

In [ ]:
for r in list(rolling)[:5]:
    print(r)

The first few values in the data are `[  NaN,  0.3, -1.2,  1.9,  0.2,  0.5, -0.1, ...]` and you can see that each window shifts one further to the right over this data. Like so:

    data:                         [  NaN,  0.3, -1.2,  1.9,  0.2,  0.5, -0.1,  ...]
    window1:    [                    NaN]  
    window2:          [              NaN,  0.3]
    window3:                [        NaN,  0.3, -1.2]
    window4:                      [  NaN,  0.3, -1.2,  1.9]
    window5:                            [  0.3, -1.2,  1.9,  0.2]

As you can see, the first three windows are not completely full as the rolling window always starts further to the left than where the data starts. So the first three windows do not actually contain 4 values, but henceforth they do.

Once you have the rolling "groups", you can apply aggregate functions to it in pretty much the same way you would with groupby: 

In [ ]:
display(rolling.mean()[10:15])
display(rolling.max()[10:15])
display(rolling.sum()[10:15])

### Exercise 31

Above you smoothed the growth of a single plant, let's now compare the two. Make a single plot showing the smoothed daily growth of the control group and of the fertilised group over the whole experiment, so that you can compare both curves.

Give the figure a title, a label on the y-axis, and a legend that names the two groups.

Tip: The easiest way to create the plot is to create two DataFrames, one for each treatment group, and once you have finished the smoothing, combine them. You can do it all in a single DataFrame, but that would require some concepts we haven't covered. 

In [ ]:
# YOUR CODE HERE